In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.patches as mpatches

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

dset_ctrl_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_ocean= xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
dset_ctrl['lead_time']=ld
dset_sens['lead_time']=ld
dset_ctrl_em['lead_time']=ld
dset_sens_em['lead_time']=ld

dset_ctrl_land['lead_time']=ld
dset_sens_land['lead_time']=ld
dset_ctrl_em_land['lead_time']=ld
dset_sens_em_land['lead_time']=ld


dset_ctrl_ocean['lead_time']=ld
dset_sens_ocean['lead_time']=ld
dset_ctrl_em_ocean['lead_time']=ld
dset_sens_em_ocean['lead_time']=ld

# Convert the dataset to a DataFrame
dset_ctrl = dset_ctrl.to_dataframe().reset_index()
dset_sens = dset_sens.to_dataframe().reset_index()
dset_ctrl_em = dset_ctrl_em.to_dataframe().reset_index()
dset_sens_em = dset_sens_em.to_dataframe().reset_index()

dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
dset_sens_land = dset_sens_land.to_dataframe().reset_index()
dset_ctrl_em_land = dset_ctrl_em_land.to_dataframe().reset_index()
dset_sens_em_land = dset_sens_em_land.to_dataframe().reset_index()

dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()
dset_ctrl_em_ocean = dset_ctrl_em_ocean.to_dataframe().reset_index()
dset_sens_em_ocean = dset_sens_em_ocean.to_dataframe().reset_index()

# --- limiti y comuni: unione dei dati dei tre pannelli (margine 10%) ---
_dfs = [dset_ctrl, dset_sens, dset_ctrl_land, dset_sens_land, dset_ctrl_ocean, dset_sens_ocean]
_ymin = min(d[var].min() for d in _dfs)
_ymax = max(d[var].max() for d in _dfs)
_margin = 0.10 * (_ymax - _ymin)
YLIM = (_ymin - _margin, _ymax + _margin)

# --- significativita' bootstrap (calcolata nel notebook di calcolo, letta qui) ---
def _load_sig(region_suffix):
    ds = xr.open_mfdataset(
        f'{SAVE_PATH}/{exp_sens}-{exp_ctrl}_{var}_lead_*_1x1_global_accsig{region_suffix}_1999.nc',
        concat_dim='lead_time', combine='nested', chunks=-1)
    return ds['sig'].values

sig_global = _load_sig('')
sig_land = _load_sig('_land')
sig_ocean = _load_sig('_ocean')

def add_significance(ax, sig):
    """Segna con '*' i lead significativi (bootstrap, dal notebook di calcolo)."""
    y0, y1 = ax.get_ylim()
    ypos = y1 - 0.05 * (y1 - y0)
    for xi in range(len(leads)):
        if sig[xi]:
            ax.text(xi, ypos, '*', ha='center', va='top', fontsize=20, fontweight='bold')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens['exp'] = 'SENS'
dset_ctrl['exp'] = 'CTRL'
df = pd.concat([dset_sens, dset_ctrl])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_global)
ax.set_title('a) GLOBAL')

fig.savefig(f'{FIG_DIR}/{var}_acc_global_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens_land['exp'] = 'SENS'
dset_ctrl_land['exp'] = 'CTRL'
df = pd.concat([dset_sens_land, dset_ctrl_land])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em_land.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em_land.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_land)
ax.set_title('b) LAND ONLY')

fig.savefig(f'{FIG_DIR}/{var}_acc_land_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens_ocean['exp'] = 'SENS'
dset_ctrl_ocean['exp'] = 'CTRL'
df = pd.concat([dset_sens_ocean, dset_ctrl_ocean])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em_ocean.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em_ocean.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_ocean)
ax.set_title('c) OCEAN ONLY')

fig.savefig(f'{FIG_DIR}/{var}_acc_ocean_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# composito 1x3: unisce global / land / ocean in un'unica figura (come Fig.3)
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

img1 = mpimg.imread(f'{FIG_DIR}/{var}_acc_global_1999.png')
img2 = mpimg.imread(f'{FIG_DIR}/{var}_acc_land_1999.png')
img3 = mpimg.imread(f'{FIG_DIR}/{var}_acc_ocean_1999.png')

fig = plt.figure(figsize=(10, 8), constrained_layout=False)
gs = GridSpec(1, 3, wspace=0.0, hspace=0.0)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
for ax, img in zip(axes, [img1, img2, img3]):
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout(pad=0.0)
plt.savefig(f'{FIG_DIR}/fig5_{var}_acc_1999.png', dpi=600, bbox_inches='tight')
